# Black-body radiation

*thermodynamics · intermediate* · [open in Tensorizer](https://jackyko1991.github.io/tensorizer/#/thermodynamics/blackbody_radiation)

Planck's law gives the energy density of thermal radiation at every wavelength. Broadcasting a column of temperatures against a row of wavelengths builds the whole $(T, \lambda)$ spectrum table in one line; argmax finds Wien's peak and torch.trapezoid integrates the spectrum to the Stefan–Boltzmann $T^4$ law.

**Vector form**

$$u(\lambda, T) = \frac{8\pi h c}{\lambda^5}\,\frac{1}{e^{hc/\lambda k T} - 1},\qquad \lambda_{\max} T = b,\qquad \int_0^\infty u\,d\lambda = \frac{4\sigma}{c}\,T^4$$

**Index (tensor) form**

$$u_{ab} = \frac{8\pi h c}{\lambda_b^5}\,\frac{1}{\exp\!\big(hc / \lambda_b k T_a\big) - 1},\qquad U_a = \sum_b \tfrac12\,(u_{ab} + u_{a,b+1})\,(\lambda_{b+1} - \lambda_b)$$

| Symbol | Code | Shape | Meaning |
|---|---|---|---|
| $T_a$ | `T[:, None]` | (A, 1) | batch of temperatures, as a column |
| $\lambda_b$ | `lam[None, :]` | (1, L) | wavelength grid, as a row |
| $u_{ab}$ | `planck(lam, T)` | (A, L) | broadcasting the column against the row fills the whole table |
| $\lambda_{\max,a}$ | `lam[u.argmax(dim=-1)]` | (A,) | Wien's peak of every spectrum at once |
| $U_a$ | `torch.trapezoid(u, lam, dim=-1)` | (A,) | repeated $b$ ⇒ summed: total energy density, $\propto T_a^4$ |
| $RGB_a$ | `approximate_color(T)` | (A, 3) | Planck at 610, 550 and 465 nm, normalised: a rough visible colour |

The wavelength grid is logarithmic from 10 nm to 100 µm, wide enough that the missing tails change the integral by less than $10^{-5}$, and the computation runs in float64 with expm1 for accuracy. Wien's constant $b \approx 2.898\times 10^{-3}$ m·K comes out of the argmax to the grid resolution (about 0.2 %). The curve colour samples the spectrum at three wavelengths standing in for red, green and blue — a crude stand-in for a real colour-matching calculation, but it shows the glow going from orange through white to pale blue. The plot is rescaled every frame: the peak height grows like $T^5$. The shaded band marks the visible range 380–750 nm.


## Setup

On Colab, choose **Runtime → Change runtime type → GPU** to run on a GPU — the code uses it automatically.

In [ ]:
import os
import subprocess
import sys

# Shared helpers (problems/_common) come from the repository; skip the clone if we are already inside it.
if not os.path.isdir("problems"):
    if not os.path.isdir("tensorizer"):
        subprocess.run(["git", "clone", "--depth", "1", "https://github.com/jackyko1991/tensorizer.git"], check=True)
    os.chdir("tensorizer")
sys.path.insert(0, os.getcwd())
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "pyyaml", "tqdm"], check=True)

import torch

print("torch", torch.__version__, "· device:", "cuda" if torch.cuda.is_available() else "cpu")


## 1. `main.py` — main loop

In [ ]:
%%writefile problems/thermodynamics/blackbody_radiation/main.py
"""Main loop: parameters → tensors → solve → plot."""

from pathlib import Path

from problems._common.runtime import cli, device

from . import equation, plot, solve, tensors


def run(params: dict, out_dir: Path) -> Path:
    dev = device()
    T = tensors.temperatures(params["T_min"], params["T_max"], params["n_temps"], dev)  # (A,)
    lam = tensors.wavelengths(dev)  # (L,)

    r = solve.solve(T, lam)
    exact = equation.stefan_boltzmann(T)  # (A,)
    for a in [0, len(T) // 2, len(T) - 1]:
        print(
            f"T = {T[a]:7.0f} K   λ_max = {1e9 * r['lam_max'][a]:6.1f} nm   λ_max T = {r['lam_max'][a] * T[a]:.4e} m K"
            f"   U = {r['U'][a]:.4e} J/m³  (4σT⁴/c = {exact[a]:.4e})"
        )
    print(f"Wien's constant b = {equation.WIEN_B:.4e} m K")
    return plot.animate(T, lam, r, out_dir / "blackbody_radiation.gif")


if __name__ == "__main__":
    cli(run, __file__)


## 2. `equation.py` — physics → tensor translation

In [ ]:
%%writefile problems/thermodynamics/blackbody_radiation/equation.py
r"""Physics → tensor translation.

Planck's law: the energy density of black-body radiation per unit wavelength is

    u(\lambda, T) = \frac{8 \pi h c}{\lambda^5} \, \frac{1}{e^{hc / \lambda k T} - 1}

For a batch of temperatures T_a and a grid of wavelengths λ_b the index form is

    u_{ab} = \frac{8 \pi h c}{\lambda_b^5} \, \frac{1}{\exp(hc / \lambda_b k T_a) - 1}

No index is summed: T_a as a column (A, 1) broadcast against λ_b as a row (1, L) gives the (A, L) table.
Two classic laws follow from it:

    Wien:              \lambda_{\max} T = b \approx 2.898 \times 10^{-3}\ \text{m K}
    Stefan–Boltzmann:  U_a = \int u_{ab} \, d\lambda_b = \frac{4 \sigma}{c} T_a^4

The integral sums over the repeated wavelength index b (torch.trapezoid along the last dimension).
"""

import math

import torch

H = 6.62607015e-34  # Planck constant (J s)
C = 2.99792458e8  # speed of light (m/s)
K_B = 1.380649e-23  # Boltzmann constant (J/K)
SIGMA = 2 * math.pi**5 * K_B**4 / (15 * H**3 * C**2)  # Stefan–Boltzmann constant ≈ 5.670e-8 W/(m² K⁴)
WIEN_B = 2.897771955e-3  # Wien's displacement constant (m K)


def planck(lam: torch.Tensor, T: torch.Tensor) -> torch.Tensor:
    r"""u_{ab} = 8πhc / λ_b⁵ / (exp(hc / λ_b k T_a) − 1)  in J/m⁴.   lam: (L,), T: (A,) → (A, L)"""
    x = H * C / (lam[None, :] * K_B * T[:, None])  # (A, L)  hc / λkT by broadcasting
    return 8 * math.pi * H * C / lam[None, :] ** 5 / torch.expm1(x)  # expm1(x) = eˣ − 1, accurate for small x


def peak_wavelength(lam: torch.Tensor, u: torch.Tensor) -> torch.Tensor:
    """λ_max of every spectrum: the grid wavelength with the largest u.   (A, L) → (A,)"""
    return lam[u.argmax(dim=-1)]


def total_energy(lam: torch.Tensor, u: torch.Tensor) -> torch.Tensor:
    r"""U_a = Σ_b ½(u_{ab} + u_{a,b+1})(λ_{b+1} − λ_b)  in J/m³.   (A, L) → (A,)"""
    return torch.trapezoid(u, lam, dim=-1)


def stefan_boltzmann(T: torch.Tensor) -> torch.Tensor:
    """The exact total, U = 4σT⁴ / c.   (A,) → (A,)"""
    return 4 * SIGMA / C * T**4


def approximate_color(T: torch.Tensor) -> torch.Tensor:
    """A rough visible colour: Planck at red, green and blue wavelengths, scaled so the brightest is 1.  (A, 3)"""
    rgb_lam = torch.tensor([610e-9, 550e-9, 465e-9], dtype=T.dtype, device=T.device)  # (3,)
    u = planck(rgb_lam, T)  # (A, 3)
    return u / u.max(dim=-1, keepdim=True).values  # (A, 3) each channel in [0, 1]


## 3. `tensors.py` — PyTorch tensors

In [ ]:
%%writefile problems/thermodynamics/blackbody_radiation/tensors.py
"""Build the concrete tensors: the batch of temperatures and the wavelength grid (float64)."""

import torch


def temperatures(T_min: float, T_max: float, n: int, device: torch.device) -> torch.Tensor:
    """T_a in kelvin, sweeping from T_min to T_max.  Shape (A,)."""
    return torch.linspace(T_min, T_max, n, dtype=torch.float64, device=device)


def wavelengths(device: torch.device, n: int = 4000) -> torch.Tensor:
    """λ_b in metres, logarithmically spaced from 10 nm to 100 µm.  Shape (L,).

    Log spacing puts many points near the short-wavelength peak and still reaches far into the long tail.
    """
    return torch.logspace(-8, -4, n, dtype=torch.float64, device=device)


## 4. `solve.py` — solver

In [ ]:
%%writefile problems/thermodynamics/blackbody_radiation/solve.py
"""Evaluate Planck's law for every temperature at once, then read off Wien's peak and the total energy."""

import torch

from .equation import approximate_color, peak_wavelength, planck, total_energy


def solve(T: torch.Tensor, lam: torch.Tensor) -> dict[str, torch.Tensor]:
    """T: (A,), lam: (L,).  Returns the spectra (A, L) and per-temperature results (A,)."""
    u = planck(lam, T)  # (A, L)
    return {
        "u": u,
        "lam_max": peak_wavelength(lam, u),  # (A,)
        "U": total_energy(lam, u),  # (A,)
        "rgb": approximate_color(T),  # (A, 3)
    }


## 5. `plot.py` — 2D/3D plot → GIF

In [ ]:
%%writefile problems/thermodynamics/blackbody_radiation/plot.py
"""Animate the spectrum while the temperature sweeps up, beside the total energy and the T⁴ law."""

from pathlib import Path

import matplotlib.pyplot as plt
import torch

from problems._common.plotting import ACCENT, frame_indices, save_gif

from .equation import WIEN_B, stefan_boltzmann

LAM_SHOWN = (100, 3000)  # wavelength window of the plot (nm)


def animate(T: torch.Tensor, lam: torch.Tensor, r: dict, path: Path) -> Path:
    temps = T.cpu().numpy()
    lam_nm = 1e9 * lam.cpu().numpy()  # (L,)
    u = 1e-6 * r["u"].cpu().numpy()  # (A, L)  J/(m³ µm)
    lam_max = 1e9 * r["lam_max"].cpu().numpy()  # (A,)
    U = r["U"].cpu().numpy()  # (A,)
    rgb = r["rgb"].cpu().numpy()  # (A, 3)
    T_line = torch.linspace(temps[0], temps[-1], 100, dtype=torch.float64)

    fig, (ax, ax_U) = plt.subplots(1, 2, figsize=(7, 4), gridspec_kw={"width_ratios": [1.5, 1]})

    # Left: the spectrum, coloured by its approximate glow, with the visible band shaded.
    ax.axvspan(380, 750, color="white", alpha=0.15, lw=0, label="visible")
    curve = ax.plot([], [], lw=2.5)[0]
    fill = [ax.fill_between(lam_nm, 0, 0)]
    peak = ax.plot([], [], "v", color="black", ms=7, label="Wien peak")[0]
    ax.set_xlim(*LAM_SHOWN)
    ax.set_xlabel("wavelength λ (nm)")
    ax.set_ylabel("u(λ, T)  (J m⁻³ µm⁻¹)")
    ax.legend(fontsize=8, loc="upper right")
    title = ax.set_title("")
    ax.set_facecolor("#1f2937")  # dark background so the glow colour stands out
    ax.grid(color="#374151")

    # Right: total energy U = ∫u dλ against T, with the exact 4σT⁴/c.
    ax_U.plot(T_line.numpy(), stefan_boltzmann(T_line).numpy(), color="gray", ls="--", label="4σT⁴/c")
    U_line = ax_U.plot([], [], color=ACCENT[1], lw=2, label="∫ u dλ (trapezoid)")[0]
    U_dot = ax_U.plot([], [], "o", color=ACCENT[1])[0]
    ax_U.set_xlim(temps[0], temps[-1])
    ax_U.set_ylim(0, 1.1 * U.max())
    ax_U.set_xlabel("T (K)")
    ax_U.set_ylabel("U (J/m³)")
    ax_U.set_title("Stefan–Boltzmann: U ∝ T⁴")
    ax_U.legend(fontsize=8, loc="upper left")

    def update(step: int):
        color = tuple(rgb[step])
        curve.set_data(lam_nm, u[step])
        curve.set_color(color)
        fill[0].remove()
        fill[0] = ax.fill_between(lam_nm, 0, u[step], color=color, alpha=0.25, lw=0)
        top = u[step].max()
        peak.set_data([lam_max[step]], [top * 1.04])
        ax.set_ylim(0, 1.15 * top)  # the peak grows like T⁵, so rescale every frame
        title.set_text(
            f"T = {temps[step]:.0f} K   λ_max = {lam_max[step]:.0f} nm   (b / T = {1e9 * WIEN_B / temps[step]:.0f})"
        )
        U_line.set_data(temps[: step + 1], U[: step + 1])
        U_dot.set_data([temps[step]], [U[step]])

    fig.tight_layout()
    return save_gif(fig, update, frame_indices(len(temps)), path)


## Run

Change the parameters (or any file above) and run again.

In [ ]:
import importlib
from pathlib import Path

from IPython.display import Image, display

from problems._common.runtime import load_meta, resolve_params

params = {
    'T_min': 3000.0,
    'T_max': 10000.0,
    'n_temps': 90
}

# Re-import so edits in the %%writefile cells above take effect.
for name in [m for m in sys.modules if m.startswith("problems.thermodynamics.blackbody_radiation")]:
    del sys.modules[name]
main = importlib.import_module("problems.thermodynamics.blackbody_radiation.main")
problem_dir = Path("problems/thermodynamics/blackbody_radiation")
output = main.run(resolve_params(load_meta(problem_dir), params), Path("out"))
display(Image(filename=str(output)))
